In [ ]:
!pip install geopandas shapely geohash2
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import box, Point
from shapely import wkt
import geohash2
from shapely.wkt import loads

  Preparing metadata (setup.py) ... done
  Created wheel for geohash2: filename=geohash2-1.1-py3-none-any.whl size=15543 sha256=7afe78c021ecba4b835a2d5769815dce7266f64f96ca27a79dab6d53d6867135
  Stored in directory: /root/.cache/pip/wheels/00/d5/b6/3fbe4088f7912982f596eaddfd593d16096468a2f13e470ae7
Successfully built geohash2


In [ ]:
df = pd.read_csv("nyc_traffic_congestion.csv")
df

,Boro,Vol,geometry,congestion,congestion_level,timestamp
0,Brooklyn,252,POLYGON ((-73.97028542436098 40.67474068522885...,0.680168,0,6/16/2025 19:45
1,Brooklyn,204,POLYGON ((-73.97028542436098 40.67474068522885...,0.136034,0,6/16/2025 20:00
2,Brooklyn,257,POLYGON ((-73.97028542436098 40.67474068522885...,0.736849,0,6/16/2025 20:15
3,Brooklyn,179,POLYGON ((-73.97028542436098 40.67474068522885...,-0.147370,0,6/16/2025 20:30
4,Brooklyn,192,POLYGON ((-73.97028542436098 40.67474068522885...,0.000000,0,6/16/2025 20:45
...,...,...,...,...,...,...
148797,Brooklyn,120,POLYGON ((-73.93800729996994 40.65260186906368...,1.349000,1,5/10/2024 16:30
148798,Brooklyn,95,POLYGON ((-73.93800729996994 40.65260186906368...,0.615848,0,5/10/2024 16:45
148799,Brooklyn,119,POLYGON ((-73.93800729996994 40.65260186906368...,1.319674,1,5/10/2024 17:00
148800,Brooklyn,114,POLYGON ((-73.93800729996994 40.65260186906368...,1.173043,1,5/10/2024 17:15


In [ ]:
df1 = pd.read_csv("traffic_related_complaints.csv")
df1

,Created Date,Closed Date,Agency,Problem (formerly Complaint Type),Problem Detail (formerly Descriptor),Location Type,Resolution Description,Location
0,12/31/2023 11:44:07 PM,01/01/2024 12:44:40 AM,NYPD,Illegal Parking,Posted Parking Sign Violation,Street/Sidewalk,The Police Department responded to the complai...,POINT (-73.948440468792 40.781460020796)
1,12/31/2023 11:43:38 PM,01/01/2024 12:44:38 AM,NYPD,Blocked Driveway,No Access,Street/Sidewalk,The Police Department responded to the complai...,POINT (-73.910570056834 40.62527066379)
2,12/31/2023 11:40:14 PM,01/01/2024 06:22:35 AM,NYPD,Illegal Parking,Commercial Overnight Parking,Street/Sidewalk,The Police Department responded to the complai...,POINT (-73.877102118195 40.714115704239)
3,12/31/2023 11:40:13 PM,01/01/2024 06:28:20 AM,NYPD,Illegal Parking,Blocked Sidewalk,Street/Sidewalk,The Police Department responded to the complai...,POINT (-73.857159979825 40.755178958687)
4,12/31/2023 11:39:48 PM,01/01/2024 03:15:41 PM,NYPD,Blocked Driveway,No Access,Street/Sidewalk,The Police Department issued a summons in resp...,POINT (-73.908552052217 40.7677225536)
...,...,...,...,...,...,...,...,...
1139656,01/01/2023 12:03:42 AM,01/01/2023 03:02:12 AM,NYPD,Illegal Parking,Blocked Hydrant,Street/Sidewalk,The Police Department responded to the complai...,POINT (-73.910374166734 40.71481064815)
1139657,01/01/2023 12:01:50 AM,01/01/2023 02:31:47 AM,NYPD,Illegal Parking,Blocked Crosswalk,Street/Sidewalk,The Police Department issued a summons in resp...,POINT (-73.956138250348 40.617894376102)
1139658,01/01/2023 12:01:39 AM,01/01/2023 12:58:18 AM,NYPD,Blocked Driveway,Partial Access,Street/Sidewalk,The Police Department reviewed your complaint ...,POINT (-73.910059895479 40.709790230832)
1139659,01/01/2023 12:00:57 AM,01/04/2023 03:09:28 PM,DOT,Street Condition,Blocked - Construction,Street,The Department of Transportation inspected the...,POINT (-73.91047850878 40.849297276034)


In [ ]:
df1['Created Date'] = pd.to_datetime(df1['Created Date'])

filtered_df = df1[
    (df1['Created Date'].dt.year >= 2024) &
    (df1['Created Date'].dt.year <= 2025)
]
filtered_df

/tmp/ipython-input-1774137431.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df1['Created Date'] = pd.to_datetime(df1['Created Date'])


,Created Date,Closed Date,Agency,Problem (formerly Complaint Type),Problem Detail (formerly Descriptor),Location Type,Resolution Description,Location
0,2023-12-31 23:44:07,01/01/2024 12:44:40 AM,NYPD,Illegal Parking,Posted Parking Sign Violation,Street/Sidewalk,The Police Department responded to the complai...,POINT (-73.948440468792 40.781460020796)
1,2023-12-31 23:43:38,01/01/2024 12:44:38 AM,NYPD,Blocked Driveway,No Access,Street/Sidewalk,The Police Department responded to the complai...,POINT (-73.910570056834 40.62527066379)
2,2023-12-31 23:40:14,01/01/2024 06:22:35 AM,NYPD,Illegal Parking,Commercial Overnight Parking,Street/Sidewalk,The Police Department responded to the complai...,POINT (-73.877102118195 40.714115704239)
3,2023-12-31 23:40:13,01/01/2024 06:28:20 AM,NYPD,Illegal Parking,Blocked Sidewalk,Street/Sidewalk,The Police Department responded to the complai...,POINT (-73.857159979825 40.755178958687)
4,2023-12-31 23:39:48,01/01/2024 03:15:41 PM,NYPD,Blocked Driveway,No Access,Street/Sidewalk,The Police Department issued a summons in resp...,POINT (-73.908552052217 40.7677225536)
...,...,...,...,...,...,...,...,...
1139656,2023-01-01 00:03:42,01/01/2023 03:02:12 AM,NYPD,Illegal Parking,Blocked Hydrant,Street/Sidewalk,The Police Department responded to the complai...,POINT (-73.910374166734 40.71481064815)
1139657,2023-01-01 00:01:50,01/01/2023 02:31:47 AM,NYPD,Illegal Parking,Blocked Crosswalk,Street/Sidewalk,The Police Department issued a summons in resp...,POINT (-73.956138250348 40.617894376102)
1139658,2023-01-01 00:01:39,01/01/2023 12:58:18 AM,NYPD,Blocked Driveway,Partial Access,Street/Sidewalk,The Police Department reviewed your complaint ...,POINT (-73.910059895479 40.709790230832)
1139659,2023-01-01 00:00:57,01/04/2023 03:09:28 PM,DOT,Street Condition,Blocked - Construction,Street,The Department of Transportation inspected the...,POINT (-73.91047850878 40.849297276034)


In [ ]:
filtered_df

,Created Date,Closed Date,Agency,Problem (formerly Complaint Type),Problem Detail (formerly Descriptor),Location Type,Resolution Description,Location


In [ ]:
df1["Problem (formerly Complaint Type)"].dt.date.value_counts().sort_index()

,count
Created Date,
2023-01-01,1997
2023-01-02,2320
2023-01-03,3266
2023-01-04,3332
2023-01-05,3233
...,...
2023-12-27,2757
2023-12-28,3085
2023-12-29,2993
